# CRR Binomial Tree Model

## 1. 연구 질문과 시간 격자

고정된 만기 $T$에서 시간 스텝 수 $N$을 늘릴 때

$$
\Delta t = \frac{T}{N}, \qquad N \to \infty, \qquad \Delta t \to 0
$$

이므로, 연구 질문은 **CRR 유럽형 옵션가격이 향후 구현할 BSM 가격에 수렴하는가?**이다. 스텝 수를 0으로 줄이는 것이 아니라 스텝 간격을 0으로 보낸다.


## 2. CRR 핵심 가정과 수식

유럽형 call/put, 연속복리 무위험이자율 $r$, 연속배당수익률 $q$, 일정한 변동성 $\sigma$를 가정한다. 한 스텝의 상승·하락 배수와 위험중립확률은

$$
u=e^{\sigma\sqrt{\Delta t}}, \qquad d=\frac{1}{u}, \qquad
p=\frac{e^{(r-q)\Delta t}-d}{u-d}
$$

이고 한 스텝 할인계수는 $e^{-r\Delta t}$이다. 만기 payoff에서 시작해

$$
V_{i,j}=e^{-r\Delta t}\left[pV_{i+1,j+1}+(1-p)V_{i+1,j}\right]
$$

로 backward induction한다. 유한한 $N$에서는 odd/even 스텝에 따른 진동이 가능하므로 매 스텝에서 가격이 단조롭게 수렴한다고 가정하지 않는다.


In [1]:
# 3. package 함수 import
import pandas as pd

from option_pricing_volatility.models.binomial import crr_price


In [2]:
# 4. 작은 synthetic call/put 예제
synthetic_parameters = {
    "spot": 100.0,
    "strike": 100.0,
    "maturity": 1.0,
    "rate": 0.05,
    "volatility": 0.2,
    "dividend_yield": 0.0,
}
example_steps = 4

example_prices = pd.DataFrame(
    [
        {
            "option_type": option_type,
            "crr_price": crr_price(
                **synthetic_parameters,
                steps=example_steps,
                option_type=option_type,
            ),
        }
        for option_type in ("call", "put")
    ]
)
example_prices


,option_type,crr_price
0,call,9.970523
1,put,5.093465


In [3]:
# 5. 수렴 실험용 tidy 결과 테이블
steps_grid = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

convergence_results = pd.DataFrame(
    [
        {
            "option_type": option_type,
            "steps": steps,
            "dt": synthetic_parameters["maturity"] / steps,
            "crr_price": crr_price(
                **synthetic_parameters,
                steps=steps,
                option_type=option_type,
            ),
        }
        for option_type in ("call", "put")
        for steps in steps_grid
    ],
    columns=["option_type", "steps", "dt", "crr_price"],
)

# 향후 BSM 구현 뒤 bsm_price, absolute_error, relative_error 열을 추가한다.
convergence_results


,option_type,steps,dt,crr_price
0,call,1,1.000000,12.162285
1,call,2,0.500000,9.540501
2,call,4,0.250000,9.970523
3,call,8,0.125000,10.205099
4,call,16,0.062500,10.326651
5,call,32,0.031250,10.388346
6,call,64,0.015625,10.419400
7,call,128,0.007812,10.434976
8,call,256,0.003906,10.442776
9,call,512,0.001953,10.446679
